# Mathematical framework NAC and NALU

The purpose of this notebook is architectural bookkeeping and verification before training or out-of-distribution (OOD) experiments. It records the mathematical definitions and conventions for the original Neural Accumulator (NAC) and Neural Arithmetic Logic Unit (NALU).

**Reference:** Trask, Hill, Reed, Rae, Dyer & Blunsom (2018), [*Neural Arithmetic Logic Units*, arXiv:1808.00508v1](https://arxiv.org/abs/1808.00508v1), **§2 and Figure 2 (pp. 2–3)**. Mistry (2023), *An Investigation into Neural Arithmetic Logic Modules*, is supplementary context, **not** the specification for this implementation.

**Implementation:** [`nalu.py`](nalu.py), containing `NAC` and `NALU`. Verification checks below adapt the supplied `nalu_test.py` for direct execution as notebook cells.

## Notation conventions and execution contract

We use **batched row vectors**, as in `nalu.py`. Capital letters denote batched matrices and parameters; lower-case $x_{bi}$ denotes one input scalar.

| Symbol | Meaning | Shape or domain |
|:--|:--|:--|
| $B$ | Batch size | positive integer |
| $I$ | Input dimension | positive integer |
| $O$ | Output dimension | positive integer |
| $X$ | Input tensor | $\mathbb R^{B\times I}$ |
| $\widehat W,\widehat M$ | Unconstrained, learnable arithmetic parameters | $\mathbb R^{I\times O}$ |
| $W$ | Effective (transformed) arithmetic matrix | $\mathbb R^{I\times O}$ |
| $G$ | Learnable input-dependent gate matrix (NALU only) | $\mathbb R^{I\times O}$ |
| $A,M,g,Y$ | Additive output, multiplicative output, gate, final output | $\mathbb R^{B\times O}$ |
| $\epsilon$ | Constant inside the logarithm | finite, strictly positive in this implementation |
| $\sigma(z)$ | Logistic sigmoid | $1/(1+e^{-z})$ |
| $\odot$ | Elementwise product | — |

In the original paper, matrix products are written for column vectors. We express the **same computations** in row-vector notation, so that $XW$ means `(B, I) @ (I, O) -> (B, O)`.

The symbol $M$ denotes the **multiplicative branch output**, while $\widehat M$ denotes a **learnable parameter matrix**; they are different quantities. In `nalu.py`, the lowercase variables are `a`, `m`, `g`, and `y`.

## 1. Neural Accumulator (NAC)

### 1.1 Effective weights and forward computation

Trask et al. parameterise the arithmetic weights as

$$
W=\tanh(\widehat W)\odot\sigma(\widehat M)
$$

and define the NAC's output by

$$
A=XW,\qquad
A_{bo}=\sum_{i=1}^{I}X_{bi}W_{io}.
$$

The transformed weights are bounded by $-1<W_{io}<1$ for finite real-valued parameters. Values near $+1$, $-1$, and $0$ have the intended interpretation of selecting an input positively, negatively, or not at all. **The parameterisation does not force exact discrete values at finite parameter settings**, nor does it guarantee that optimisation learns them.

There is **no learnable bias and no output activation**. The NAC parameter count is

$$N_{\mathrm{NAC}}=2IO.$$

**Code correspondence — `NAC.forward()` in `nalu.py`:**

```python
W = self.W_hat.tanh() * self.M_hat.sigmoid()
a = x @ W
```

### 1.2 Analytical NAC example with finite parameters

Let $B=O=1$, $I=2$ and choose

$$
X=\begin{bmatrix}2&-4\end{bmatrix},\quad
\widehat W=\begin{bmatrix}\operatorname{atanh}(1/2)\\\operatorname{atanh}(-1/2)\end{bmatrix},\quad
\widehat M=\begin{bmatrix}0\\0\end{bmatrix}.
$$

Since $\sigma(0)=1/2$,

$$
W=\begin{bmatrix}1/4\\-1/4\end{bmatrix},\qquad
A=2(1/4)+(-4)(-1/4)=\boxed{1.5}.
$$

This is a test of the continuous equation, **not** an example of perfectly discrete $\{-1,0,1\}$ weights.

In [ ]:
import math
import torch
from torch.func import functional_call
from nalu import NAC, NALU

# Test-only choice, not an epsilon prescribed by Trask et al.
EPS = 0.125
DTYPE = torch.float64

hat_w = torch.tensor([[math.atanh(0.5)], [-math.atanh(0.5)]], dtype=DTYPE)
hat_m = torch.zeros_like(hat_w)
nac = NAC(hat_w, hat_m)
x = torch.tensor([[2.0, -4.0]], dtype=DTYPE)
a, d = nac(x, diagnostics=True)
torch.testing.assert_close(d['W'], torch.tensor([[0.25], [-0.25]], dtype=DTYPE), rtol=0, atol=1e-14)
torch.testing.assert_close(a, torch.tensor([[1.5]], dtype=DTYPE), rtol=0, atol=1e-14)
print('PASS — finite-parameter NAC example: W = [0.25, -0.25], A = 1.5')

## 2. Neural Arithmetic Logic Unit (NALU)

### 2.1 Shared weights, two arithmetic branches, and gate

The NALU uses the **same** effective arithmetic matrix $W$ for both branches:

$$
\begin{aligned}
W &=\tanh(\widehat W)\odot\sigma(\widehat M),\\
A &=XW,\\
M &=\exp\!\left(\log(|X|+\epsilon)W\right),\\
g &=\sigma(XG),\\
\boxed{Y}&=g\odot A+(1-g)\odot M.
\end{aligned}
$$

Every logarithm, absolute value, exponential, and sigmoid is applied **elementwise**; matrix multiplication is performed only where juxtaposition appears. Both branches return $(B,O)$ matrices and are mixed coordinatewise by the $(B,O)$ gate. There are **no bias vectors**.

**Code correspondence — `NALU.forward()` in `nalu.py`:**

```python
W = self.W_hat.tanh() * self.M_hat.sigmoid()
a = x @ W
m = ((x.abs() + self.eps).log() @ W).exp()
g = (x @ self.G).sigmoid()
y = g * a + (1 - g) * m
```

**Three non-negotiable features of the original architecture:** (i) **one tied $W$** for both branches; (ii) **input-dependent**, bias-free $g=\sigma(XG)$; (iii) **bias-free arithmetic paths** with no output nonlinearity. The NALU has **$3IO$** trainable scalars in $\widehat W,\widehat M,G$ rather than two independently parameterised arithmetic cells.

### 2.2 Equivalent multiplicative interpretation and its domain

For a single example $b$ and output coordinate $o$, the multiplicative branch is

$$
M_{bo}=\exp\left(\sum_{i=1}^I W_{io}\log(|X_{bi}|+\epsilon)\right)
=\prod_{i=1}^I(|X_{bi}|+\epsilon)^{W_{io}}.
$$

Thus coefficients near $+1$ multiply input magnitudes, coefficients near $-1$ introduce reciprocal factors, and coefficients near $0$ suppress factors. The **$\epsilon$-shift means that even at idealised integer-valued weights, the computed factors are $|x_i|+\epsilon$, not exactly $|x_i|$**. It prevents $\log(0)$, but does not make division by small values numerically benign.

Since the multiplicative branch uses $|X|$ and exponentiation, $M_{bo}>0$ for finite inputs/parameters where the expression is numerically defined. **It discards input signs and cannot produce negative multiplicative outputs.** These are properties of the 2018 equations, not missing features to repair in this baseline.

### 2.3 Analytically predictable NALU example

Choose $B=O=1$, $I=2$, $X=[2,4]$, the same $\widehat W$ and $\widehat M$ as the NAC example, and

$$
G=\begin{bmatrix}(\log3)/2\\0\end{bmatrix},\qquad \epsilon=1/8.
$$

Then $W=[1/4,-1/4]^\top$ and

$$
\begin{aligned}
A &=2(1/4)+4(-1/4)=-1/2,\\
M &=\left(\frac{2+1/8}{4+1/8}\right)^{1/4},\\
g &=\sigma(\log3)=3/4,\\
Y &=\frac34\left(-\frac12\right)+\frac14\left(\frac{2+1/8}{4+1/8}\right)^{1/4}.
\end{aligned}
$$

The numerical check evaluates these scalar formulas independently of the module's matrix expressions.

In [ ]:
G = torch.tensor([[math.log(3) / 2], [0.0]], dtype=DTYPE)
example = NALU(hat_w, hat_m, G, eps=EPS)
x = torch.tensor([[2.0, 4.0]], dtype=DTYPE)
y, d = example(x, diagnostics=True)
expected_m = ((2.0 + EPS) / (4.0 + EPS)) ** 0.25
expected_y = 0.75 * (-0.5) + 0.25 * expected_m
for name, expected in [('a', -0.5), ('m', expected_m), ('g', 0.75)]:
    torch.testing.assert_close(d[name], torch.tensor([[expected]], dtype=DTYPE), rtol=0, atol=1e-14)
torch.testing.assert_close(y, torch.tensor([[expected_y]], dtype=DTYPE), rtol=0, atol=1e-14)
print(f'PASS — NALU example: a=-0.5, m={expected_m:.9f}, g=0.75, y={expected_y:.9f}')

### 2.4 Structural consequences of the original gate

For finite parameters, $0<g_{bo}<1$; exact selection $g\in\{0,1\}$ is therefore a limiting idealisation. Because the gate **has no bias** and is computed directly from the input,

$$
\boxed{g(-X)=\sigma(-XG)=1-g(X)}
$$

and for $X=0$,

$$
\boxed{g(0)=\tfrac12}.
$$

In fact, at the all-zero input, $A=0$ while $M_o=\epsilon^{\sum_i W_{io}}$, giving

$$
\boxed{Y_o(0)=\tfrac12\epsilon^{\sum_i W_{io}}>0}
$$

in exact arithmetic for $\epsilon>0$ and finite parameters. This is a **deduction from the original equations**, not a claim made explicitly in Trask et al. It illustrates why a model capable of approximating arithmetic may still have systematic errors at particular inputs. Do not 'fix' this in the original baseline.

In [ ]:
x0 = torch.zeros((1, 2), dtype=DTYPE)
y0, d0 = example(x0, diagnostics=True)
assert torch.equal(d0['g'], torch.full_like(d0['g'], 0.5))
expected_m0 = EPS ** float(d0['W'].sum().detach())
torch.testing.assert_close(y0, torch.tensor([[0.5 * expected_m0]], dtype=DTYPE))
_, opposite = example(-x, diagnostics=True)
torch.testing.assert_close(opposite['g'], 1 - d['g'])
print('PASS — bias-free gate symmetry and all-zero input consequences')

## 3. Constructor assumptions, diagnostics, and learnable state

**Source vs implementation distinction.** Trask et al. specify the formula and the presence of $\epsilon$ but do **not** prescribe a complete random-initialisation procedure or a numerical value for $\epsilon$ in §2. Accordingly, the `nalu.py` constructors **require explicit tensors** for $\widehat W,\widehat M$ (and $G$ for NALU), and `NALU` additionally requires `eps > 0`. These constructor-interface choices are ours, not claims about the authors' original code.

The module clones the supplied tensors into trainable `nn.Parameter` objects. `eps` is a Python float: it is **not** a learned parameter. Both modules keep the original forward formulas unaltered, without buffers, biases, clipping, or extra regularisers.

**Diagnostic contract:** `forward(x)` returns the output only. With `diagnostics=True`, `NAC` returns `(a, {'W': W})`, while `NALU` returns `(y, {'W': W, 'a': a, 'm': m, 'g': g})`. Values retain autograd connections and are **not** cached as mutable module state.

**Expected parameter counts:** NAC $2IO$ and NALU $3IO$. For $I=3,O=2$, these are $12$ and $18$ trainable scalars, respectively.

## 4. Differentiability and parameter-gradient structure

The effective weight transform has entrywise derivatives

$$
\begin{aligned}
\frac{\partial W_{io}}{\partial\widehat W_{io}}&=\bigl(1-\tanh^2\widehat W_{io}\bigr)\sigma(\widehat M_{io}),\\
\frac{\partial W_{io}}{\partial\widehat M_{io}}&=\tanh(\widehat W_{io})\sigma(\widehat M_{io})\bigl(1-\sigma(\widehat M_{io})\bigr).
\end{aligned}
$$

For a fixed example and output coordinate, differentiating the NALU output gives

$$
\begin{aligned}
\frac{\partial Y_{bo}}{\partial W_{io}}
&=g_{bo}X_{bi}+(1-g_{bo})M_{bo}\log(|X_{bi}|+\epsilon),\\
\frac{\partial Y_{bo}}{\partial G_{io}}
&=(A_{bo}-M_{bo})g_{bo}(1-g_{bo})X_{bi}.
\end{aligned}
$$

**The first expression includes gradient contributions from both branches through the *shared* $W$.** Gradients of $\widehat W$ and $\widehat M$ then follow by the chain rule. These derivatives presume the relevant computation is finite. Since $|X|$ is nonsmooth at $X_{bi}=0$, numerical `gradcheck` must use inputs away from zero; forward equation checks can still include zero.

We verify PyTorch differentiation with `torch.autograd.gradcheck` in double precision, checking input **and** parameter derivatives. This is a local differentiability check, **not** evidence of optimisation convergence or OOD extrapolation.

## 5. Executable checks adapted from `nalu_test.py`

The following cells preserve the coverage and input values of the supplied test notebook, while removing `pytest` fixtures/parametrisation so that **Run All** executes the checks directly. The six shape cases, two formula cases, two parameter/gradient cases, and two numerical gradient cases comprise the original **12 check cases**. The analytical examples above are additional equation-level cross-checks, not new architectural features.

The checks make no assumptions that finite $\widehat W,\widehat M,G$ yield exactly discrete $W$ or $g$.

In [ ]:
# Shapes and diagnostic tensor dimensions: 6 cases (NAC/NALU × 3 shapes).
for module in (NAC, NALU):
    for shape in ((1, 1), (3, 1), (3, 2)):
        I, O = shape
        hW = torch.full(shape, 0.2)
        hM = torch.full(shape, -0.3)
        layer = NAC(hW, hM) if module is NAC else NALU(hW, hM, hW, eps=EPS)
        result, diag = layer(torch.ones(4, I), diagnostics=True)
        assert result.shape == (4, O)
        assert diag['W'].shape == (I, O)
        if module is NALU:
            assert all(diag[key].shape == (4, O) for key in ('a', 'm', 'g'))
print('PASS — 6 shape cases')

In [ ]:
# Direct formula/diagnostic checks: 2 cases (NAC and NALU).
X = torch.tensor([[0.7, -1.2, 2.0], [-0.8, 1.6, -2.3]], dtype=DTYPE)
W_hat = torch.tensor([[0.2, -0.5], [0.7, 0.3], [-0.4, 0.6]], dtype=DTYPE)
M_hat = torch.tensor([[-0.3, 0.8], [0.5, -0.7], [0.9, 0.1]], dtype=DTYPE)
Gate = torch.tensor([[0.4, -0.2], [-0.6, 0.7], [0.3, 0.5]], dtype=DTYPE)
X_with_zero = torch.cat((X, torch.zeros(1, 3, dtype=DTYPE)))

for layer in (NAC(W_hat, M_hat), NALU(W_hat, M_hat, Gate, eps=EPS)):
    output, diag = layer(X_with_zero, diagnostics=True)
    W_ref = torch.tanh(W_hat) * torch.sigmoid(M_hat)
    a_ref = X_with_zero @ W_ref
    torch.testing.assert_close(diag['W'], W_ref)
    assert diag['W'].requires_grad
    if isinstance(layer, NAC):
        assert set(diag) == {'W'}
        torch.testing.assert_close(output, a_ref)
    else:
        m_ref = torch.exp(torch.log(torch.abs(X_with_zero) + EPS) @ W_ref)
        g_ref = torch.sigmoid(X_with_zero @ Gate)
        assert set(diag) == {'W', 'a', 'm', 'g'}
        for name, expected in (('a', a_ref), ('m', m_ref), ('g', g_ref)):
            torch.testing.assert_close(diag[name], expected)
            assert diag[name].requires_grad
        torch.testing.assert_close(output, g_ref * a_ref + (1 - g_ref) * m_ref)
        _, neg_diag = layer(-X_with_zero, diagnostics=True)
        torch.testing.assert_close(neg_diag['g'], 1 - g_ref)
        torch.testing.assert_close(diag['g'][-1], torch.full((2,), 0.5, dtype=DTYPE))
        assert not torch.allclose(diag['g'][0], diag['g'][1])
    torch.testing.assert_close(layer(X_with_zero), output)
print('PASS — 2 exact-formula/diagnostic cases (including zero input)')

In [ ]:
# Registered parameters, ownership, and backward gradients: 2 cases.
for layer in (NAC(W_hat, M_hat), NALU(W_hat, M_hat, Gate, eps=EPS)):
    expected = {'W_hat': W_hat, 'M_hat': M_hat}
    if isinstance(layer, NALU):
        expected['G'] = Gate
    params = dict(layer.named_parameters())
    assert set(params) == set(expected)
    assert not dict(layer.named_buffers())
    assert sum(p.numel() for p in params.values()) == (3 if isinstance(layer, NALU) else 2) * 3 * 2
    layer(X).square().sum().backward()
    for name, p in params.items():
        assert isinstance(p, torch.nn.Parameter) and p.requires_grad
        torch.testing.assert_close(p, expected[name])
        assert p.data_ptr() != expected[name].data_ptr()
        assert p.grad is not None and torch.isfinite(p.grad).all()
# A valid zero gradient is not a failure of differentiability.
print('PASS — 2 parameter-count/registration/backpropagation cases')

In [ ]:
# Numerical gradcheck for input and every trainable parameter: 2 cases.
# All input entries here are nonzero, avoiding the kink in abs(x).
for layer in (NAC(W_hat, M_hat), NALU(W_hat, M_hat, Gate, eps=EPS)):
    input_tensor = X.clone().requires_grad_()
    names, params = zip(*layer.named_parameters())
    def forward(x, *p):
        return functional_call(layer, dict(zip(names, p)), (x,))
    assert torch.autograd.gradcheck(forward, (input_tensor, *params))
print('PASS — 2 torch.autograd.gradcheck cases')

## 6. Verification boundaries and next questions

The executable checks establish agreement with the explicitly stated 2018 computations, shapes and parameterisation, plus correctness of local derivatives at the tested points. They **do not** establish reliable convergence to target arithmetic weights or numerical extrapolation.

In particular, distinguish:

1. **Representability:** whether the architecture can implement or approximate a target arithmetic relation on its intended domain;
2. **Learnability:** whether optimisation reaches the requisite effective weights and gate behaviour;
3. **Extrapolation:** whether the learned function remains accurate outside the training distribution.

These distinctions will matter in later scratchpad experiments, but **no training distributions, optimisers, losses, or experiment-specific assumptions are specified in this architecture notebook**.

**Source provenance:** All NAC/NALU equations above are from Trask et al. (2018), §2; the finite-parameter examples, parameter counts, gate identities, and derivative expressions are mathematical deductions or verification constructions based on those equations. Mistry (2023) is reserved for later comparative discussion, not used to modify the baseline.